# Signpost-D one-pass: a decision model trained on our own cases

Run 7 settings: LoRA r 64 (run 6: 16) and a deeper scoring head (`HEAD_EXTRA` residual blocks, each starting at zero so the published head is unchanged at step 0).

Signpost answers grounded questions well, but on [Typed Decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) it scored **0.4175** (notebook 06), below the majority baseline. Its weakness there is *judgement*: what should happen next, how serious or urgent something is, inferred from structured information.

**Signpost-D one-pass** starts from **Signpost one-pass's weights** ([`skamalj/signpost-one-pass`](https://huggingface.co/skamalj/signpost-one-pass): merged encoder and scoring head, with a new LoRA), trained on **our own decision cases** only. `07` runs 2–4 started from plain ModernBERT-large and stayed at a uniform guess; set `START` to `answerdotai/ModernBERT-large` for that start.
- written by subagents from a plain-English brief (`data/signpost_d/brief_v2.md`), never from the benchmark;
- **soft-labelled** by three independent labellers (cautious, pragmatic, neutral; averaged);
- checked by validators, then put in the published **System One** form (`data/signpost_d/format_spec.md`): `noul` (true/false), `choice` (labels), `score` (an ordered rubric, lowest first, keyed `"0"`…`"n-1"`).

Our cases and the benchmark's are rendered for the model by **one** function, `render`, so the model sees both in the same way.

**One pass per question** (the design of Signpost one-pass, `05_one_pass_signpost.ipynb`, and its loader `signpost/signpost_one_pass.py`), instead of `07`'s one pass per option:

```text
user: <state> [SEP] current agent: none [SEP] question: Q [SEP] option: A. … [SEP] option: B. … [SEP] …
   → ModernBERT-large + LoRA (one pass) → the mean of each option's own tokens → one shared head → softmax over the options
```

Why (runs 2 and 3 of `07` stayed at a uniform guess): in `07` each option's input was the whole state plus that option, mean-pooled, so options differed only in a few tokens out of hundreds and started out scored almost alike. Here each option is pooled over **its own tokens only**, and the options **see each other** inside the encoder (how many levels a scale has, what the alternatives are), which also matters for spreading probability honestly. Score levels keep their rubric order (lowest first, as published); `choice` and `noul` options are shuffled in training.

Signpost-D is a **decision model only**: no routing data in training, and no routing tests. Routing stays with the original Signpost (Signpost-R).

**Measured:**
1. our **held-out Signpost-D cases** (accuracy against the label's top option, KL, Brier, ECE) after each epoch. These choose the epoch and the temperature;
2. **grounded-question tests** (E12, the independent and independent-2 question sets) after training: the general reading check;
3. the **benchmark test split, once**, at the end, as published (T = 1) and with one temperature per question type fitted on our dev cases;
4. a **calibration check** on both: whether the predictions sit closer to 0 and 1 than the gold (top-probability histogram, entropy, and how much of the KL comes from options the model rules out but the gold does not).

Nothing is uploaded. One T4 is enough.

In [1]:
# Cell 1 — Setup: packages, settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, math, os, random, shutil, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch loads)
from collections import Counter
from pathlib import Path

import pandas as pd
import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

D_SET = "v1_1"                        # data/signpost_d/<D_SET>/systemone.jsonl (v1: 300 pilot + 1,200 cases; v1_1: v1 + 150 invoice cases, I1-I3)
START = "skamalj/signpost-one-pass"  # the weights we start from (encoder/ + head.pt); "answerdotai/ModernBERT-large" = plain, new head
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.1  # run 7: r 64 (run 6: 16), same alpha / r
HEAD_EXTRA = 2                        # residual blocks added to the head (0 = Signpost's head as published)
LR, HEAD_LR = 2e-4, 5e-4             # as 07 from Signpost; use HEAD_LR 1e-3 for a new head (03, 05)
EPOCHS = 3                            # the best epoch on the dev cases is kept (07 run 5: best at 2, worse by 4)
BATCH = 8                             # questions per step, one encoder pass each (with gradient checkpointing)
MAX_LEN = 1536                        # state + question + every option in one input (ours ~600 tokens at most; as 05, room for 50 options)
MAX_OPTIONS = 50                      # the grounded-question test independent 2 has up to 50 options (System One allows 26 per choice)
SCORE_DIST_W = 0.5                    # score questions: weight of |expected level - target expected level| / (levels - 1)
DEV_FRACTION = 0.15
SEED = 42                             # the first seed; a collapsed start retries with SEED + 1000, + 2000
COLLAPSE_STEP, COLLAPSE_MARGIN, ATTEMPTS = 400, 0.02, 3   # at this step, dev KL must be this far below a uniform guess's
BENCH = "LocalLLaMA/typed-decisions"  # scored once at the end; never used for training or selection
SEP, ROUTE_Q = " [SEP] ", "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("signpost_d_onepass_run.log")
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()
REV = {START: api.model_info(START).sha, BENCH: api.dataset_info(BENCH).sha}


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | {DEVICE} | "
    f"start {START} @ {REV[START][:8]}")

[03:40:43] torch 2.11.0+cu128 | transformers 5.17.0 | peft 0.21.2 | cuda:0 | start skamalj/signpost-one-pass @ 0cb2271d


In [2]:
# Cell 2a — The Signpost-D cases (System One form): one item per question, with the labellers' averaged distribution as the target

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():                  # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def render(q):
    """A System One question -> ({candidate name: description}, the gold's keys in the same order). One rendering for our
    cases and the benchmark: noul as true/false with their descriptions, choice by its labels, score by its criteria
    strings, lowest first (keys "0".."n-1")."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    assert len(opts) == len(keys), q
    return opts, keys


d_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / D_SET / "systemone.jsonl")]
def dev_split(ids):
    ids = sorted(ids)
    random.Random(SEED).shuffle(ids)
    return set(ids[: int(len(ids) * DEV_FRACTION)])


ADDED = ("I",)                            # cases added after v1 (invoices): split on their own, so v1's dev cases stay as in run 6
dev_ids = dev_split(c["id"] for c in d_cases if not c["id"].startswith(ADDED)) | \
          dev_split(c["id"] for c in d_cases if c["id"].startswith(ADDED))


def d_items(c):
    out = []
    for qid, q in c["questions"].items():
        opts, keys = render(q)
        gold = c["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "id": c["id"], "qid": qid, "domain": c["meta"]["domain"],
                    "messages": [{"role": "user", "content": state_text(c["state"])}], "current": None,
                    "question": q["instructions"], "options": opts, "levels": list(opts) if q["type"] == "score" else None,
                    "soft": {n: gold[k] for n, k in zip(opts, keys)}})
    return out


train_d = [it for c in d_cases if c["id"] not in dev_ids for it in d_items(c)]
dev_d = [it for c in d_cases if c["id"] in dev_ids for it in d_items(c)]
print(f"Signpost-D ({D_SET}): {len(d_cases)} cases | train {len(train_d)} questions | dev {len(dev_d)} questions "
      f"({len(dev_ids)} cases) | {dict(Counter(it['task'] for it in train_d))}")

Cloning into 'llms-the-hard-way'...


Signpost-D (v1_1): 1650 cases | train 6669 questions | dev 1155 questions (247 cases) | {'d_noul': 2359, 'd_choice': 2281, 'd_score': 2029}


In [3]:
# Cell 2b — Grounded-question tests (from Signpost): the general reading check

load = lambda p: json.load(open(DATA_DIR / "modernbert_router" / p))
e12 = load("choice_test/items.json")["items"]
ind_q, ind2_q = load("independent/questions.json")["items"], load("independent2/questions.json")["items"]
print(f"E12 {len(e12)} | independent questions {len(ind_q)} | independent 2 questions {len(ind2_q)} "
      f"(up to {max(len(q['options']) for q in ind2_q)} options)")

E12 170 | independent questions 300 | independent 2 questions 200 (up to 50 options)


In [4]:
# Cell 3 — The model: Signpost one-pass's encoder (START) + a new LoRA, one pass per question, each option pooled over its own
# tokens, the published scoring head with HEAD_EXTRA residual blocks (05's OnePassModel; input format as signpost/signpost_one_pass.py)

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer

FROM_SIGNPOST = START != "answerdotai/ModernBERT-large"
START_DIR = Path(snapshot_download(START, revision=REV[START], allow_patterns=None if FROM_SIGNPOST else ["*.json", "*.safetensors", "*.txt"]))
ENC_DIR = START_DIR / "encoder" if FROM_SIGNPOST else START_DIR


class ScoreHead(nn.Module):
    """Signpost's scoring head (Dropout, Linear(H, 256), GELU, Dropout, Linear(256, 1)) with `extra` residual blocks of 256
    units between its two layers. Each block's last layer starts at zero, so a head loaded from Signpost starts out exactly as
    published, and the blocks then learn on top of it."""

    def __init__(self, hidden, extra=0):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, 256), nn.GELU())
        self.blocks = nn.ModuleList([nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 256), nn.GELU(), nn.Linear(256, 256))
                                     for _ in range(extra)])
        for b in self.blocks:
            nn.init.zeros_(b[3].weight); nn.init.zeros_(b[3].bias)
        self.out = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))

    def forward(self, x):
        h = self.inp(x)
        for b in self.blocks:
            h = h + b(h)
        return self.out(h)

    def load_published(self, sd):
        """Signpost's head (a plain Sequential: layers 1 and 4) into this head."""
        self.inp[1].load_state_dict({"weight": sd["1.weight"], "bias": sd["1.bias"]})
        self.out[1].load_state_dict({"weight": sd["4.weight"], "bias": sd["4.bias"]})


class Model:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(ENC_DIR)
        self.tok.padding_side = "right"
        enc = AutoModel.from_pretrained(ENC_DIR, dtype=torch.float32).to(DEVICE)
        enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})   # recompute, don't store, activations
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=["Wqkv", "Wo", "Wi"]))
        self.head = ScoreHead(self.H, HEAD_EXTRA).to(DEVICE)
        if FROM_SIGNPOST:
            self.head.load_published(torch.load(START_DIR / "head.pt", map_location="cpu")["state_dict"])   # one-pass's head

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    @staticmethod
    def build(item):
        """messages SEP current agent SEP question SEP option: A SEP option: B ... -> (text, character span of each option)."""
        text = SEP.join([f"{m['role']}: {m['content']}" for m in item["messages"]]
                        + [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"])
        spans = []
        for name, desc in item["options"].items():
            seg = f"{name}. {desc}" if desc else name
            text += SEP + "option: "
            spans.append((len(text), len(text) + len(seg)))
            text += seg
        return text, spans

    def fit(self, item):
        """The whole input must fit MAX_LEN: a case has one message, so nothing can be dropped (as the loader)."""
        assert len(item["options"]) <= MAX_OPTIONS, f"{len(item['options'])} options > MAX_OPTIONS={MAX_OPTIONS}"
        text, spans = self.build(item)
        n = len(self.tok(text)["input_ids"])
        assert n <= MAX_LEN, f"input of {n} tokens > MAX_LEN={MAX_LEN}"
        return text, spans

    def scores(self, items):
        """-> [items, most options] scores (missing options -1e4): one encoder pass per question, each option the mean of its
        own tokens."""
        built = [self.fit(it) for it in items]
        enc = self.tok([t for t, _ in built], padding=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")                                            # [B, T, 2] characters of each token
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).last_hidden_state
        K = max(len(sp) for _, sp in built)
        spans = torch.tensor([sp + [(0, 0)] * (K - len(sp)) for _, sp in built])     # [B, K, 2]; missing options are empty
        s, e = offsets[..., 0], offsets[..., 1]                                        # [B, T]
        real = enc["attention_mask"].bool() & (e > s)                                  # real, non-special tokens
        member = (real[:, None] & (s[:, None] < spans[..., 1:]) & (e[:, None] > spans[..., :1])).float().to(DEVICE)   # [B, K, T]
        counts = member.sum(-1)                                                        # [B, K] tokens per option
        assert int((counts > 0).sum()) == sum(len(sp) for _, sp in built), "an option got no tokens"
        pooled = (member @ hidden.float()) / counts.clamp(min=1).unsqueeze(-1)         # [B, K, H] mean of each option's tokens
        return self.head(pooled).squeeze(-1).masked_fill(counts == 0, -1e4)

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
# Cell 4a — The loss: soft labels (+ an expected-level term for score questions); metrics on the dev cases
# (accuracy against the label's top option, KL, Brier, ECE)

def soft_loss(scores, items):
    """Cross-entropy against the labellers' distribution (= KL + a constant). Score questions add
    SCORE_DIST_W x |expected level - target expected level| / (levels - 1), each option weighted by its level in the
    rubric (it["levels"]), not by its place after the options are shuffled."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        lvl = torch.tensor([[it["levels"].index(n) if it["levels"] else 0 for n in it["options"]] + [0] * (K - len(it["options"]))
                            for it in items], device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * lvl).sum(1) - (target * lvl).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    return loss


def d_predictions(model, items, batch=16):
    """-> [(item, {option: probability})] for Signpost-D items."""
    model.train(False)
    out = []
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            p = torch.softmax(model.scores(chunk), dim=1).float().cpu()
            out += [(it, {n: p[i, j].item() for j, n in enumerate(it["options"])}) for i, it in enumerate(chunk)]
    return out


def soften(p, temperature):
    """Probabilities softened as p ** (1 / T), renormalised."""
    q = [max(v, 1e-12) ** (1 / temperature) for v in p]
    z = sum(q)
    return [v / z for v in q]


def t_of(temperature, task):
    """A temperature: one number, or one per question type ({"d_noul": T, ...})."""
    return temperature.get(task, 1.0) if isinstance(temperature, dict) else temperature


def d_metrics(preds, temperature=1.0):
    """The benchmark's four measures, against our own soft labels: accuracy (top option = the label's top option), KL from the
    label, Brier, ECE (10 bins). With a temperature (one, or one per question type), probabilities are softened."""
    rows = []
    for it, p in preds:
        q = {n: max(v, 1e-12) ** (1 / t_of(temperature, it["task"])) for n, v in p.items()}
        z = sum(q.values())
        q = {n: v / z for n, v in q.items()}
        g = it["soft"]
        ok = max(q, key=q.get) == max(g, key=g.get)
        rows.append({"task": it["task"], "ok": ok, "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    return {"all": summ(rows), **{t: summ([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperatures(preds):
    """One temperature per question type, each minimising KL on our own dev cases of that type (never the benchmark)."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    out = {}
    for task in sorted({it["task"] for it, _ in preds}):
        sub = [(it, p) for it, p in preds if it["task"] == task]
        out[task] = min(grid, key=lambda t: d_metrics(sub, t)["all"]["kl"])
    return out


def calibration(pairs):
    """pairs: [(predicted probabilities, gold probabilities)], same option order. Do the predictions sit nearer 0 and 1 than
    the gold? Top-probability histograms side by side, mean normalised entropy, and the over-certain options: predicted
    under 0.01 where the gold has 0.05 or more, with the share of the total KL they cause (scorer's floor 1e-9)."""
    edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0001]
    hist = lambda tops: [sum(lo <= t < hi for t in tops) for lo, hi in zip(edges, edges[1:])]
    ent = lambda v: -sum(x * math.log(x) for x in v if x > 0) / math.log(len(v)) if len(v) > 1 else 0.0
    n_opt = over = 0
    kl_all = kl_over = 0.0
    for p, g in pairs:
        for pi, gi in zip(p, g):
            n_opt += 1
            term = gi * math.log(gi / max(pi, 1e-9)) if gi > 0 else 0.0
            kl_all += term
            if pi < 0.01 and gi >= 0.05:
                over += 1
                kl_over += term
    return {"top_p_bins": [f"{lo:.2f}-{min(hi, 1):.2f}" for lo, hi in zip(edges, edges[1:])],
            "pred_top_hist": hist([max(p) for p, _ in pairs]), "gold_top_hist": hist([max(g) for _, g in pairs]),
            "pred_entropy": round(statistics.mean(ent(p) for p, _ in pairs), 4),
            "gold_entropy": round(statistics.mean(ent(g) for _, g in pairs), 4),
            "over_certain_options": over, "of_options": n_opt,
            "kl_share_from_over_certain": round(kl_over / kl_all, 4) if kl_all else 0.0}

In [6]:
# Cell 4b — The training loop: shuffled batches of Signpost-D questions; after each epoch the dev cases are scored and the
# epoch with the lowest dev KL is kept. Collapse guard: from plain ModernBERT-large a start can stay at chance (Signpost's
# run 22; run 3 here), so at COLLAPSE_STEP the dev KL on 300 questions must beat a uniform guess by COLLAPSE_MARGIN.

class Collapsed(Exception):
    """The predictions stayed at a uniform guess; retry with another seed."""


def uniform_kl(items):
    """KL of a uniform guess from the labels: the level a collapsed model sits at."""
    return statistics.mean(sum(g * math.log(g * len(it["soft"])) for g in it["soft"].values() if g > 0) for it in items)


def batches_for_epoch(epoch, seed):
    d = train_d[:]
    random.Random(seed + epoch).shuffle(d)
    return [d[i:i + BATCH] for i in range(0, len(d), BATCH)]


def train_model(model, seed):
    per_epoch = len(batches_for_epoch(1, seed))
    check = dev_d[:300]
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * per_epoch
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0, history, run = (float("inf"), None, 0), 0, time.perf_counter(), [], []
    log(f"training: {per_epoch} batches per epoch, {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        for items in batches_for_epoch(epoch, seed):
            items = [dict(it) for it in items]
            for it in items:                                    # choice and noul options shuffled; score levels keep their order
                if it["task"] != "d_score":
                    names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            loss = soft_loss(model.scores(items), items)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss {statistics.mean(run[-100:]):.4f} | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
            if step == COLLAPSE_STEP:
                kl, base = d_metrics(d_predictions(model, check))["all"]["kl"], uniform_kl(check)
                log(f"  collapse check at step {step}: dev KL {kl:.4f} vs uniform {base:.4f}")
                if kl > base - COLLAPSE_MARGIN:
                    raise Collapsed(f"dev KL {kl:.4f} at step {step}, uniform guess {base:.4f}")
                model.train(True)
        dev = d_metrics(d_predictions(model, dev_d))["all"]
        history.append({"epoch": epoch, **dev})
        log(f"epoch {epoch} done | dev {dev}")
        if dev["kl"] < best[0]:
            best = (dev["kl"], {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (dev KL {best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": best[2], "history": history}

In [7]:
# Cell 5a — Timing and the question test (from Signpost's 03)

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": [{"role": "user", "content": it["state"]}], "current": None,
                                    "question": it["question"], "options": {o: "" for o in it["options"]}})
        top = int(p.argmax())
        rows.append({"type": it["type"], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [8]:
# Cell 5b — The grounded-question tests on a model: R = {test: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}


def evaluate(model):
    model.train(False)
    R, LAT = {}, {}
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        rows, LAT[name] = questions(model, items)
        R[f"{name} questions"] = (sum(r["ok"] for r in rows), len(rows))
        for t in TYPE:
            rs = [r for r in rows if r["type"] == t]
            if rs:
                R[f"{name} {TYPE[t]}"] = (sum(r["ok"] for r in rs), len(rs))
        log(f"{name} questions {R[f'{name} questions'][0]}/{len(rows)} | " +
            " · ".join(f"{TYPE[t]} {R[f'{name} {TYPE[t]}'][0]}/{R[f'{name} {TYPE[t]}'][1]}" for t in TYPE if f"{name} {TYPE[t]}" in R))
    return R, LAT

In [9]:
# Cell 5c — The benchmark, scored once at the end (general mode: never used for training or for any choice above).
# Scoring copied from the benchmark's bench/score_with_jobs.py (score_one, 10-bin ECE).

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":
        pred = dict(pred)
        pred["probabilities"] = {{"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()}
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p, am_g = p.index(max(p)), g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def bench_item(case, q):
    """A benchmark question as a Signpost item, through the same render() as our own cases (state as its JSON text, as
    /v1/systemone sends it)."""
    opts, keys = render(q)
    return {"messages": [{"role": "user", "content": state_text(case["state"])}], "current": None,
            "question": q.get("instructions") or "", "options": opts}, keys


def bench_score(model, cases, temperature=1.0, pairs=None):
    """The four measures on the benchmark. temperature: one number, or one per question type. pairs (a list), if given,
    collects (prediction, gold) in gold order for calibration()."""
    model.train(False)
    rows = []
    for case in cases:
        for qid, q in case["questions"].items():
            item, keys = bench_item(case, q)
            p = soften(model.probs(item).tolist(), t_of(temperature, "d_" + q["type"]))
            pred = {"probabilities": dict(zip(keys, p))}
            ok, kl, brier, conf = score_one(pred, case["gold"][qid])
            rows.append({"type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
            if pairs is not None:
                g = case["gold"][qid]["probabilities"]
                pairs.append(([pred["probabilities"].get(k, 0.0) for k in g], [float(v) for v in g.values()]))
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    out = {"all": summ(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    return out

In [10]:
# Cell 6 — Train (no 'before' measures), then every measure

if AMP and torch.cuda.memory_allocated(DEVICE) > 1e9:
    raise RuntimeError("the GPU already holds an earlier model: restart the kernel (or start a new session) and run all cells "
                       "once, from the top")
attempts = []
for attempt in range(ATTEMPTS):
    seed = SEED + 1000 * attempt
    random.seed(seed); torch.manual_seed(seed)
    model = Model()
    if attempt == 0:
        model.enc.print_trainable_parameters()
    log(f"attempt {attempt + 1} (seed {seed})")
    try:
        trained = train_model(model, seed)
        attempts.append({"seed": seed, "status": "ok"})
        break
    except Collapsed as e:
        log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
        attempts.append({"seed": seed, "status": f"collapsed: {e}"})
        model = None; gc.collect()
        if AMP:
            torch.cuda.empty_cache()
else:
    raise RuntimeError(f"training collapsed in every attempt: {attempts}")

dev_preds = d_predictions(model, dev_d)
T = fit_temperatures(dev_preds)                            # one per question type, on our dev cases
AFTER = {"T=1": d_metrics(dev_preds), "T per type": d_metrics(dev_preds, T)}
for k, v in AFTER.items():
    log(f"after training, dev cases, {k}: " + " | ".join(f"{t} {m}" for t, m in v.items()))
for name, part in (("v1 cases (as run 6)", [(it, p) for it, p in dev_preds if not it["id"].startswith(ADDED)]),
                   ("added cases (invoices)", [(it, p) for it, p in dev_preds if it["id"].startswith(ADDED)])):
    if part:
        log(f"after training, dev {name}, T per type: {d_metrics(part, T)['all']}")

R, LAT = evaluate(model)                                   # grounded-question tests after training (compare Signpost in the ledger)
log(f"temperatures (dev): {T}")
TEST = bench_split("test")
pairs_t1, pairs_t = [], []
BENCH_RES = {"T=1": bench_score(model, TEST, 1.0, pairs_t1), "T per type": bench_score(model, TEST, T, pairs_t)}
CALIB = {"dev, T=1": calibration([([p[n] for n in it["options"]], [it["soft"][n] for n in it["options"]]) for it, p in dev_preds]),
         "benchmark, T=1": calibration(pairs_t1), "benchmark, T per type": calibration(pairs_t)}
for k, v in CALIB.items():
    log(f"calibration, {k}: {v}")
for k, v in BENCH_RES.items():
    log(f"benchmark test, {k}: {v['all']}")
    for g, m in v.items():
        if g != "all":
            print(f"  {g:38s} acc {m['accuracy']:.3f}  KL {m['kl_from_gold']:.3f}  Brier {m['brier']:.3f}  ECE {m['ece']:.3f}  (n {m['n']})")
print("benchmark reference points (accuracy): uniform 0.308 · majority ~0.47 · Signpost as published 0.4175 (06) · teacher 0.735")
json.dump({"d_set": D_SET, "revisions": REV, **trained, "attempts": attempts, "dev_after": AFTER, "temperature": T, "calibration": CALIB,
           "question_tests": {k: list(v) for k, v in R.items()}, "latency": {k: lat_summary(v) for k, v in LAT.items()},
           "benchmark": BENCH_RES}, open(f"signpost_d_onepass_results_{D_SET}.json", "w"), indent=1)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

trainable params: 28,786,688 || all params: 423,568,384 || trainable%: 6.7962
[03:41:10] attempt 1 (seed 42)
[03:41:10] training: 834 batches per epoch, 2502 steps


/tmp/ipykernel_116/1365511475.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[03:43:07]   epoch 1 step 100/2502 | loss 1.2967 | ~47.1 min left | GPU 2.1 GB (peak 3.1)
[03:45:08]   epoch 1 step 200/2502 | loss 1.0255 | ~45.7 min left | GPU 2.1 GB (peak 3.1)
[03:47:10]   epoch 1 step 300/2502 | loss 1.0098 | ~44.1 min left | GPU 2.1 GB (peak 3.1)
[03:49:12]   epoch 1 step 400/2502 | loss 0.9415 | ~42.3 min left | GPU 2.1 GB (peak 3.1)
[03:49:24]   collapse check at step 400: dev KL 0.4226 vs uniform 0.6641
[03:51:23]   epoch 1 step 500/2502 | loss 0.9222 | ~40.9 min left | GPU 2.1 GB (peak 3.1)
[03:53:26]   epoch 1 step 600/2502 | loss 0.9383 | ~38.9 min left | GPU 2.1 GB (peak 3.1)
[03:55:26]   epoch 1 step 700/2502 | loss 0.9049 | ~36.8 min left | GPU 2.1 GB (peak 3.1)
[03:57:28]   epoch 1 step 800/2502 | loss 0.9283 | ~34.7 min left | GPU 2.1 GB (peak 3.1)
[03:58:59] epoch 1 done | dev {'n': 1155, 'accuracy': 0.6693, 'kl': 0.4946, 'brier': 0.321, 'ece': 0.0518}
[04:00:19]   epoch 2 step 900/2502 | loss 0.8219 | ~34.1 min left | GPU 2.2 GB (peak 3.2)
[04:02:19]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

[04:39:32] calibration, dev, T=1: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [198, 182, 165, 152, 202, 120, 136], 'gold_top_hist': [17, 32, 36, 51, 149, 296, 574], 'pred_entropy': 0.6287, 'gold_entropy': 0.2925, 'over_certain_options': 0, 'of_options': 3601, 'kl_share_from_over_certain': 0.0}
[04:39:32] calibration, benchmark, T=1: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [447, 213, 214, 272, 523, 236, 95], 'gold_top_hist': [366, 497, 398, 232, 203, 104, 200], 'pred_entropy': 0.6261, 'gold_entropy': 0.6391, 'over_certain_options': 0, 'of_options': 7100, 'kl_share_from_over_certain': 0.0}
[04:39:32] calibration, benchmark, T per type: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [498, 297, 242, 441, 492, 30, 0], 'gold_top_hist': [366, 497, 398, 

In [ ]:
# Cell 7 — Save signpost-d-onepass-<D_SET>/ in Signpost one-pass format (encoder merged, head, config, signpost_one_pass.py)
# and a zip; nothing is uploaded

OUT = Path(f"signpost-d-onepass-{D_SET}")
shutil.rmtree(OUT, ignore_errors=True)
model.enc.save_pretrained(OUT / "adapter")
merged = model.enc.merge_and_unload()
merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
model.tok.save_pretrained(OUT / "encoder")
torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H,
            "extra_blocks": HEAD_EXTRA}, OUT / "head.pt")
json.dump({"name": "Signpost-D one-pass", "format": "one-pass", "start": START, "start_revision": REV[START], "sep": SEP,
           "max_len": MAX_LEN, "max_options": MAX_OPTIONS, "route_question": ROUTE_Q, "temperature": T, "d_set": D_SET,
           "best_epoch": trained["best_epoch"], "seed_attempts": attempts, "lora_r": LORA_R, "head_extra_blocks": HEAD_EXTRA, "lora_alpha": LORA_ALPHA},
          open(OUT / "signpost_config.json", "w"), indent=1)
shutil.copy(DATA_DIR.parent / "signpost" / "signpost_one_pass.py", OUT / "signpost_one_pass.py")
shutil.make_archive(str(OUT), "zip", root_dir=".", base_dir=str(OUT))
log(f"saved {OUT}/ and {OUT}.zip ({Path(str(OUT) + '.zip').stat().st_size / 1e9:.2f} GB) | download it if the result is worth keeping")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Reading the results

- **Dev cases by epoch:** does the data teach the skill on unseen cases of our own? (Signpost as published scored 0.398 on the pilot's dev cases.)
- **Grounded-question tests:** general reading, compared with Signpost's scores in the ledger.
- **Calibration:** if the predicted top-probability histogram piles up above 0.95 while the gold's does not, and over-certain options cause much of the KL, the model sits too near 0 and 1; per-type temperatures should pull it back.
- **Benchmark:** a general-mode measure (our data only; the benchmark never chose anything). Compare with Signpost (0.4175) and run 1 (0.3965).